# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## 1. Setup & Imports

In [1]:
%pip install numpy

%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121   # for GPU support

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

2.5.1+cu121
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## 2. Hyperparameters

In [3]:
BATCH_SIZE = 256
NUM_EPOCHS = 50
NUM_WORKERS = 8

## 3. Data Loading

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

Files already downloaded and verified
Files already downloaded and verified
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## 4. Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## 5. Training & Evaluation Helpers

`train_one_epoch` and `validate` each handle a single pass through their respective loader.  
`fit` orchestrates the loop, tracks history, and restores the best checkpoint.

In [6]:
criterion = nn.CrossEntropyLoss()


def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> tuple[float, float]:
    """Run one full pass over `loader` in training mode.

    Returns
    -------
    train_loss : float  – mean cross-entropy loss over all samples
    train_acc  : float  – accuracy in percent
    """
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode.

    Returns
    -------
    val_loss : float  – mean cross-entropy loss over all samples
    val_acc  : float  – accuracy in percent
    """
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = NUM_EPOCHS,
) -> dict[str, list[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    best_val_acc = 0.0
    best_state   = None
    history: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train(model, train_loader, optimizer)
        val_loss, val_acc = validate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
            f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
            f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
        )

    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    return history

## 6. Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [7]:
model_a    = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=1e-2)

print("=== Experiment A: SGD + LeakyReLU ===")
history_a = fit(model_a, optimizer_a, train_loader, test_loader)

=== Experiment A: SGD + LeakyReLU ===
Epoch  1/50 | train loss 2.2985, train acc 14.73% | val loss 2.2925, val acc 19.52%
Epoch  2/50 | train loss 2.2809, train acc 19.30% | val loss 2.2617, val acc 19.54%
Epoch  3/50 | train loss 2.2184, train acc 21.01% | val loss 2.1543, val acc 24.07%
Epoch  4/50 | train loss 2.1027, train acc 24.37% | val loss 2.0448, val acc 26.86%
Epoch  5/50 | train loss 2.0050, train acc 27.81% | val loss 1.9443, val acc 30.06%
Epoch  6/50 | train loss 1.9098, train acc 31.12% | val loss 1.8574, val acc 32.91%
Epoch  7/50 | train loss 1.8275, train acc 34.60% | val loss 1.7859, val acc 36.31%
Epoch  8/50 | train loss 1.7543, train acc 37.05% | val loss 1.7199, val acc 38.43%
Epoch  9/50 | train loss 1.6947, train acc 39.19% | val loss 1.6459, val acc 40.94%
Epoch 10/50 | train loss 1.6370, train acc 41.01% | val loss 1.6010, val acc 42.13%
Epoch 11/50 | train loss 1.5883, train acc 43.04% | val loss 1.5622, val acc 43.86%
Epoch 12/50 | train loss 1.5441, train

## 7. Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [8]:
model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=1e-3)

print("=== Experiment B: Adam + LeakyReLU ===")
history_b = fit(model_b, optimizer_b, train_loader, test_loader)

=== Experiment B: Adam + LeakyReLU ===
Epoch  1/50 | train loss 1.5229, train acc 44.61% | val loss 1.2271, val acc 55.86%
Epoch  2/50 | train loss 1.1256, train acc 60.08% | val loss 1.1304, val acc 59.46%
Epoch  3/50 | train loss 0.9446, train acc 66.86% | val loss 0.9183, val acc 68.01%
Epoch  4/50 | train loss 0.8115, train acc 71.58% | val loss 0.8734, val acc 69.27%
Epoch  5/50 | train loss 0.7210, train acc 74.83% | val loss 0.7873, val acc 72.27%
Epoch  6/50 | train loss 0.6325, train acc 78.07% | val loss 0.7630, val acc 73.37%
Epoch  7/50 | train loss 0.5621, train acc 80.43% | val loss 0.7566, val acc 74.05%
Epoch  8/50 | train loss 0.4935, train acc 82.86% | val loss 0.7789, val acc 74.64%
Epoch  9/50 | train loss 0.4276, train acc 85.18% | val loss 0.7667, val acc 75.16%
Epoch 10/50 | train loss 0.3638, train acc 87.35% | val loss 0.7651, val acc 75.85%
Epoch 11/50 | train loss 0.2931, train acc 89.99% | val loss 0.8301, val acc 74.69%
Epoch 12/50 | train loss 0.2417, trai

## 8. Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [9]:
model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=1e-3)

print("=== Experiment C: Adam + Tanh ===")
history_c = fit(model_c, optimizer_c, train_loader, test_loader)

=== Experiment C: Adam + Tanh ===
Epoch  1/50 | train loss 1.3971, train acc 49.90% | val loss 1.1144, val acc 60.55%
Epoch  2/50 | train loss 1.0008, train acc 64.70% | val loss 0.9530, val acc 66.74%
Epoch  3/50 | train loss 0.8376, train acc 70.71% | val loss 0.8658, val acc 69.19%
Epoch  4/50 | train loss 0.7156, train acc 75.25% | val loss 0.8131, val acc 71.88%
Epoch  5/50 | train loss 0.6136, train acc 78.94% | val loss 0.7893, val acc 72.97%
Epoch  6/50 | train loss 0.5226, train acc 81.90% | val loss 0.7780, val acc 73.38%
Epoch  7/50 | train loss 0.4290, train acc 85.66% | val loss 0.7717, val acc 74.06%
Epoch  8/50 | train loss 0.3429, train acc 88.93% | val loss 0.7794, val acc 74.00%
Epoch  9/50 | train loss 0.2639, train acc 92.03% | val loss 0.8094, val acc 74.79%
Epoch 10/50 | train loss 0.1989, train acc 94.52% | val loss 0.8301, val acc 74.57%
Epoch 11/50 | train loss 0.1385, train acc 96.82% | val loss 0.8516, val acc 74.85%
Epoch 12/50 | train loss 0.0920, train acc